In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics.pairwise import cosine_similarity


# Only using jobs
## Tune a global Threshold
### Settiing up main functions

In [3]:

def parse_skill_list(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return []

    seen = set()
    skills = []
    for raw in text.split("|"):
        skill = raw.strip()
        if skill and skill not in seen:
            seen.add(skill)
            skills.append(skill)
    return skills


parse_skill_list(value)
Takes a string like:

"Excel | Auditing | Tax"
and converts it into:

["Excel", "Auditing", "Tax"]

In [4]:
def load_jsonl(path):
    records = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    if not records:
        raise ValueError(f"No records found in {path}")
    return records

load_jsonl(path)
Reads a .jsonl file, where each line is one JSON object, and returns a list of records.

In [5]:
def build_indicator_matrix(skill_lists, skill_names):
    skill_to_idx = {skill: idx for idx, skill in enumerate(skill_names)}
    indicator = np.zeros((len(skill_lists), len(skill_names)), dtype=np.uint8)
    for row_idx, skills in enumerate(skill_lists):
        for skill in skills:
            indicator[row_idx, skill_to_idx[skill]] = 1
    return indicator


build_indicator_matrix(skill_lists, skill_names)
Creates the ground-truth label matrix.

Example:

if there are 3 skills: ["Excel", "Tax", "Audit"]
and a job has ["Excel", "Audit"]
then its row becomes:

[1, 0, 1]

In [6]:
def _safe_div(numerator, denominator):
    numerator = np.asarray(numerator, dtype=np.float64)
    denominator = np.asarray(denominator, dtype=np.float64)
    result = np.zeros_like(numerator, dtype=np.float64)
    np.divide(numerator, denominator, out=result, where=denominator != 0)
    return result


def _binary_f1(tp, fp, fn):
    precision = _safe_div(tp, tp + fp)
    recall = _safe_div(tp, tp + fn)
    return _safe_div(2.0 * precision * recall, precision + recall)


def _micro_metrics_from_counts(tp, fp, fn):
    precision = float(tp / (tp + fp)) if (tp + fp) else 0.0
    recall = float(tp / (tp + fn)) if (tp + fn) else 0.0
    f1 = float((2.0 * precision * recall) / (precision + recall)) if (precision + recall) else 0.0
    return precision, recall, f1

Metric helpers

np.divide(numerator, denominator, out=result, where=denominator != 0)
This is the key line.

It means:

- divide numerator / denominator
- store the answer in result
- but only at positions where denominator != 0

So if denominator is 0 somewhere, NumPy skips that division entirely. No warning, no inf, no nan. That entry in result stays 0.0

_binary_f1(tp, fp, fn)
Computes F1 from true positives, false positives, false negatives.

_micro_metrics_from_counts(tp, fp, fn)
Computes micro precision, recall, and F1 from global counts.

### Threshold Selection Logic

Global threshold

_find_best_global_threshold(similarity, labels)
tries one cutoff for the entire matrix.

The matrix similarity has one score for every (job, skill) pair.
The matrix labels has the true answer for every (job, skill) pair:

- 1 = this job really has this skill
- 0 = it does not

For each candidate threshold: pred = similarity >= threshold

That creates a full prediction matrix of True/False.

Then it counts:

- tp: predicted 1 and true 1
- fp: predicted 1 but true 0
- fn: predicted 0 but true 1

From those, it computes:

- micro precision
- micro recall
- micro F1

It also computes per-skill F1 and averages them into macro F1.

Then _is_better_candidate() decides which threshold is best, using this priority:

1. higher micro F1
2. if tied, higher recall
3. if tied, higher macro F1
4. if tied, lower threshold

So the code is mostly optimizing for overall micro F1.

Why try only unique similarity values?

Because predictions only change when the threshold passes one of the actual scores.

Example: if scores are [0.2, 0.5, 0.9], then threshold 0.51 gives the same predictions as 0.89. So it is enough to test unique score values.

In [7]:
EPS = 1e-12

def _is_better_candidate(candidate, incumbent):
    c_f1, c_recall, c_macro_f1, c_threshold = candidate
    i_f1, i_recall, i_macro_f1, i_threshold = incumbent

    if c_f1 > i_f1 + EPS:
        return True
    if i_f1 > c_f1 + EPS:
        return False

    if c_recall > i_recall + EPS:
        return True
    if i_recall > c_recall + EPS:
        return False

    if c_macro_f1 > i_macro_f1 + EPS:
        return True
    if i_macro_f1 > c_macro_f1 + EPS:
        return False

    return c_threshold < i_threshold - EPS


def _find_best_global_threshold(similarity, labels):
    flat_scores = similarity.ravel()
    flat_labels = labels.ravel().astype(bool)
    candidates = np.unique(flat_scores)

    best = None
    best_threshold = None

    for threshold in candidates:
        pred = flat_scores >= threshold
        tp = int(np.logical_and(pred, flat_labels).sum())
        fp = int(np.logical_and(pred, ~flat_labels).sum())
        fn = int(np.logical_and(~pred, flat_labels).sum())
        _, micro_recall, micro_f1 = _micro_metrics_from_counts(tp, fp, fn)

        pred_2d = similarity >= threshold
        tp_cols = np.logical_and(pred_2d, labels == 1).sum(axis=0)
        fp_cols = np.logical_and(pred_2d, labels == 0).sum(axis=0)
        fn_cols = np.logical_and(~pred_2d, labels == 1).sum(axis=0)
        macro_f1 = float(_binary_f1(tp_cols, fp_cols, fn_cols).mean())

        candidate = (micro_f1, micro_recall, macro_f1, float(threshold))
        if best is None or _is_better_candidate(candidate, best):
            best = candidate
            best_threshold = float(threshold)

    return best_threshold

### Training & Evaluation Pipeline

Step 1: Load data
- reads the job CSV with uuid, title, extracted_skills
- reads job embeddings JSONL
- reads skill embeddings JSONL

Step 3: Build matrices
- job_matrix: job embeddings
- skill_matrix: skill embeddings
- similarity: cosine similarity matrix
- labels: binary true-label matrix

Step 4: Select threshold
It finds one best global threshold and applies it to all skills.

Step 5: Final evaluation

After applying the global threshold, it computes:

- micro precision / recall / F1
- macro precision / recall / F1
- and stores them in metrics_df.

Step 6: Build outputs

It returns 3 DataFrames:

`metrics_df`
overall evaluation metrics

`thresholds_df`
per-skill table showing the shared global threshold and each skill's positive support

`predictions_df`
- per-job actual vs predicted skills
- counts of TP / FP / FN

In [8]:
def fit_acc_similarity(
    job_csv="data/acc/audit_tax_accounting_jobs.csv",
    job_embeddings_jsonl="embedding/jobs_train_embeddings.jsonl",
    skill_embeddings_jsonl="embedding/acc_skills_embeddings.jsonl",
):
    labels_df = pd.read_csv(job_csv, usecols=["uuid", "title", "extracted_skills"])

    job_records = load_jsonl(job_embeddings_jsonl)
    skill_records = load_jsonl(skill_embeddings_jsonl)

    jobs_df = pd.DataFrame(
        {
            "job_id": [record["job_id"] for record in job_records],
            "embedded_title": [record.get("title", "") for record in job_records],
            "job_embedding": [record["embedding"] for record in job_records],
        }
    )

    merged = jobs_df.merge(
        labels_df,
        left_on="job_id",
        right_on="uuid",
        how="left",
    )

    skill_names = [record["skill_name"] for record in skill_records]

    actual_skill_lists = merged["extracted_skills"].map(parse_skill_list).tolist()

    job_matrix = np.vstack(merged["job_embedding"].to_numpy()).astype(np.float32)
    skill_matrix = np.vstack([record["embedding"] for record in skill_records]).astype(np.float32)
    similarity = cosine_similarity(job_matrix, skill_matrix)

    labels = build_indicator_matrix(actual_skill_lists, skill_names)
    if similarity.shape != labels.shape:
        raise ValueError(
            f"Similarity matrix shape {similarity.shape} does not match label matrix shape {labels.shape}"
        )

    global_threshold = _find_best_global_threshold(similarity, labels)
    thresholds = np.full(len(skill_names), global_threshold, dtype=np.float32)
    positive_support = labels.sum(axis=0).astype(int)

    predictions = similarity >= global_threshold
    positives = labels == 1
    tp_cols = np.logical_and(predictions, positives).sum(axis=0).astype(np.int64)
    fp_cols = np.logical_and(predictions, ~positives).sum(axis=0).astype(np.int64)
    fn_cols = np.logical_and(~predictions, positives).sum(axis=0).astype(np.int64)

    total_tp = int(tp_cols.sum())
    total_fp = int(fp_cols.sum())
    total_fn = int(fn_cols.sum())

    micro_precision, micro_recall, micro_f1 = _micro_metrics_from_counts(total_tp, total_fp, total_fn)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    metrics_df = pd.DataFrame(
        [
            {
                "num_jobs": len(merged),
                "num_skills": len(skill_names),
                "global_threshold": global_threshold,
                "micro_precision": micro_precision,
                "micro_recall": micro_recall,
                "micro_f1": micro_f1,
                "macro_precision": float(macro_precision),
                "macro_recall": float(macro_recall),
                "macro_f1": float(macro_f1),
            }
        ]
    )

    thresholds_df = pd.DataFrame(
        {
            "skill_name": skill_names,
            "threshold": thresholds.astype(float),
            "positive_support": positive_support,
        }
    )

    predicted_skill_lists = [
        [skill_names[idx] for idx, is_on in enumerate(row) if is_on]
        for row in predictions
    ]
    predictions_df = pd.DataFrame(
        {
            "job_id": merged["job_id"],
            "title": merged["title"].fillna(merged["embedded_title"]).astype(str),
            "actual_skills": [" | ".join(skills) for skills in actual_skill_lists],
            "predicted_skills": [" | ".join(skills) for skills in predicted_skill_lists],
            "tp": [len(set(a) & set(p)) for a, p in zip(actual_skill_lists, predicted_skill_lists)],
            "fp": [len(set(p) - set(a)) for a, p in zip(actual_skill_lists, predicted_skill_lists)],
            "fn": [len(set(a) - set(p)) for a, p in zip(actual_skill_lists, predicted_skill_lists)],
        }
    )

    print(f"Evaluated {len(merged)} jobs against {len(skill_names)} ACC skills.")
    print(metrics_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
    print(f"\nGlobal threshold applied to all skills: {global_threshold:.4f}")

    return metrics_df, thresholds_df, predictions_df


### Final Excecution Block

runs the whole pipeline.

In [9]:
metrics_df, thresholds_df, predictions_df = fit_acc_similarity(
    job_csv="../data/acc/audit_tax_accounting_jobs.csv",
    job_embeddings_jsonl="../embedding/jobs_train_embeddings.jsonl",
    skill_embeddings_jsonl="../embedding/acc_skills_embeddings.jsonl",
)

# Optional inspection
print("\nPredictions preview:")
print(predictions_df.head(5).to_string(index=False))

Evaluated 59 jobs against 229 ACC skills.
 num_jobs  num_skills  global_threshold  micro_precision  micro_recall  micro_f1  macro_precision  macro_recall  macro_f1
       59         229            0.6489           0.1375        0.1244    0.1306           0.0412        0.0196    0.0179

Global threshold applied to all skills: 0.6489

Predictions preview:
                          job_id                                                            title                                                                                                                                                                                                                                                                                                                                                                                     actual_skills                                                                                                                                                                 

## Tune the threshold for common skill

After obtaining the global threshold, tuning will be done to obtain threshold for each skill that appears more often in the jobs. What this means is: 
for each skill column, count how many jobs truly have that skill labeled as 1.

So the decision is:

if a skill appears in at least `min_positives_for_skill_tuning` jobs, it gets its own threshold
otherwise, it just keeps the global threshold

Other than that, the rest of the code is generally the same as above. I just added more checks:
1. Prevents KeyError if a skill appears in extracted_skills but is not in the skill embedding file. 

`if skill in skill_to_idx:
    indicator[row_idx, skill_to_idx[skill]] = 1`

2. Added check for jobs with missing labels after merge

3. Added one more threshold candidate above max score. This allows the model to consider predicting nothing for a skill if that is actually best.


Currently, min_positives_for_skill_tuning=10

If your dataset is small, you might even increase it to 15 or 20 so rare skills do not overfit too easily.

In [10]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics.pairwise import cosine_similarity


def build_indicator_matrix(skill_lists, skill_names):
    skill_to_idx = {skill: idx for idx, skill in enumerate(skill_names)}
    indicator = np.zeros((len(skill_lists), len(skill_names)), dtype=np.uint8)

    for row_idx, skills in enumerate(skill_lists):
        for skill in skills:
            if skill in skill_to_idx:
                indicator[row_idx, skill_to_idx[skill]] = 1

    return indicator


EPS = 1e-12


def _is_better_candidate(candidate, incumbent):
    c_f1, c_recall, c_macro_f1, c_threshold = candidate
    i_f1, i_recall, i_macro_f1, i_threshold = incumbent

    if c_f1 > i_f1 + EPS:
        return True
    if i_f1 > c_f1 + EPS:
        return False

    if c_recall > i_recall + EPS:
        return True
    if i_recall > c_recall + EPS:
        return False

    if c_macro_f1 > i_macro_f1 + EPS:
        return True
    if i_macro_f1 > c_macro_f1 + EPS:
        return False

    return c_threshold < i_threshold - EPS


def _find_best_global_threshold(similarity, labels):
    flat_scores = similarity.ravel()
    flat_labels = labels.ravel().astype(bool)

    candidates = np.unique(flat_scores)
    candidates = np.r_[candidates, flat_scores.max() + 1e-9]

    best = None
    best_threshold = None

    for threshold in candidates:
        pred = flat_scores >= threshold
        tp = int(np.logical_and(pred, flat_labels).sum())
        fp = int(np.logical_and(pred, ~flat_labels).sum())
        fn = int(np.logical_and(~pred, flat_labels).sum())
        _, micro_recall, micro_f1 = _micro_metrics_from_counts(tp, fp, fn)

        pred_2d = similarity >= threshold
        tp_cols = np.logical_and(pred_2d, labels == 1).sum(axis=0)
        fp_cols = np.logical_and(pred_2d, labels == 0).sum(axis=0)
        fn_cols = np.logical_and(~pred_2d, labels == 1).sum(axis=0)
        macro_f1 = float(_binary_f1(tp_cols, fp_cols, fn_cols).mean())

        candidate = (micro_f1, micro_recall, macro_f1, float(threshold))
        if best is None or _is_better_candidate(candidate, best):
            best = candidate
            best_threshold = float(threshold)

    return best_threshold


def _find_best_threshold_for_one_skill(scores, labels, default_threshold):
    candidates = np.unique(scores)
    candidates = np.r_[candidates, scores.max() + 1e-9]

    best = None
    best_threshold = float(default_threshold)

    for threshold in candidates:
        pred = scores >= threshold

        tp = int(np.logical_and(pred, labels == 1).sum())
        fp = int(np.logical_and(pred, labels == 0).sum())
        fn = int(np.logical_and(~pred, labels == 1).sum())

        precision = float(tp / (tp + fp)) if (tp + fp) else 0.0
        recall = float(tp / (tp + fn)) if (tp + fn) else 0.0
        f1 = float((2.0 * precision * recall) / (precision + recall)) if (precision + recall) else 0.0

        candidate = (f1, recall, -float(threshold))
        if best is None or candidate > best:
            best = candidate
            best_threshold = float(threshold)

    return best_threshold


def fit_acc_similarity(
    job_csv="data/acc/audit_tax_accounting_jobs.csv",
    job_embeddings_jsonl="embedding/jobs_train_embeddings.jsonl",
    skill_embeddings_jsonl="embedding/acc_skills_embeddings.jsonl",
    min_positives_for_skill_tuning=10,
):
    labels_df = pd.read_csv(job_csv, usecols=["uuid", "title", "extracted_skills"])

    job_records = load_jsonl(job_embeddings_jsonl)
    skill_records = load_jsonl(skill_embeddings_jsonl)

    jobs_df = pd.DataFrame(
        {
            "job_id": [record["job_id"] for record in job_records],
            "embedded_title": [record.get("title", "") for record in job_records],
            "job_embedding": [record["embedding"] for record in job_records],
        }
    )

    merged = jobs_df.merge(
        labels_df,
        left_on="job_id",
        right_on="uuid",
        how="left",
    )

    if merged["uuid"].isna().any():
        missing_count = int(merged["uuid"].isna().sum())
        raise ValueError(f"{missing_count} embedded jobs have no matching labels in the CSV")

    skill_names = [record["skill_name"] for record in skill_records]
    actual_skill_lists = merged["extracted_skills"].map(parse_skill_list).tolist()

    job_matrix = np.vstack(merged["job_embedding"].to_numpy()).astype(np.float32)
    skill_matrix = np.vstack([record["embedding"] for record in skill_records]).astype(np.float32)

    similarity = cosine_similarity(job_matrix, skill_matrix)

    labels = build_indicator_matrix(actual_skill_lists, skill_names)
    if similarity.shape != labels.shape:
        raise ValueError(
            f"Similarity matrix shape {similarity.shape} does not match label matrix shape {labels.shape}"
        )

    global_threshold = _find_best_global_threshold(similarity, labels)

    positive_support = labels.sum(axis=0).astype(int)
    thresholds = np.full(len(skill_names), global_threshold, dtype=np.float32)
    tuning_type = np.array(["global"] * len(skill_names), dtype=object)

    for j in range(len(skill_names)):
        if positive_support[j] >= min_positives_for_skill_tuning:
            thresholds[j] = _find_best_threshold_for_one_skill(
                scores=similarity[:, j],
                labels=labels[:, j],
                default_threshold=global_threshold,
            )
            tuning_type[j] = "skill_specific"

    predictions = similarity >= thresholds[np.newaxis, :]
    positives = labels == 1

    tp_cols = np.logical_and(predictions, positives).sum(axis=0).astype(np.int64)
    fp_cols = np.logical_and(predictions, ~positives).sum(axis=0).astype(np.int64)
    fn_cols = np.logical_and(~predictions, positives).sum(axis=0).astype(np.int64)

    total_tp = int(tp_cols.sum())
    total_fp = int(fp_cols.sum())
    total_fn = int(fn_cols.sum())

    micro_precision, micro_recall, micro_f1 = _micro_metrics_from_counts(total_tp, total_fp, total_fn)

    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    metrics_df = pd.DataFrame(
        [
            {
                "num_jobs": len(merged),
                "num_skills": len(skill_names),
                "global_threshold": global_threshold,
                "micro_precision": micro_precision,
                "micro_recall": micro_recall,
                "micro_f1": micro_f1,
                "macro_precision": float(macro_precision),
                "macro_recall": float(macro_recall),
                "macro_f1": float(macro_f1),
                "min_positives_for_skill_tuning": min_positives_for_skill_tuning,
                "num_skill_specific_thresholds": int((tuning_type == "skill_specific").sum()),
            }
        ]
    )

    thresholds_df = pd.DataFrame(
        {
            "skill_name": skill_names,
            "threshold": thresholds.astype(float),
            "positive_support": positive_support,
            "tuning_type": tuning_type,
            "tp": tp_cols,
            "fp": fp_cols,
            "fn": fn_cols,
            "precision": _safe_div(tp_cols, tp_cols + fp_cols),
            "recall": _safe_div(tp_cols, tp_cols + fn_cols),
            "f1": _binary_f1(tp_cols, fp_cols, fn_cols),
        }
    )

    predicted_skill_lists = [
        [skill_names[idx] for idx, is_on in enumerate(row) if is_on]
        for row in predictions
    ]

    predictions_df = pd.DataFrame(
        {
            "job_id": merged["job_id"],
            "title": merged["title"].fillna(merged["embedded_title"]).astype(str),
            "actual_skills": [" | ".join(skills) for skills in actual_skill_lists],
            "predicted_skills": [" | ".join(skills) for skills in predicted_skill_lists],
            "tp": [len(set(a) & set(p)) for a, p in zip(actual_skill_lists, predicted_skill_lists)],
            "fp": [len(set(p) - set(a)) for a, p in zip(actual_skill_lists, predicted_skill_lists)],
            "fn": [len(set(a) - set(p)) for a, p in zip(actual_skill_lists, predicted_skill_lists)],
        }
    )

    print(f"Evaluated {len(merged)} jobs against {len(skill_names)} ACC skills.")
    print(metrics_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
    print(f"\nGlobal threshold: {global_threshold:.4f}")
    print(
        f"Skill-specific thresholds used for "
        f"{int((tuning_type == 'skill_specific').sum())} skills "
        f"(minimum positive support = {min_positives_for_skill_tuning})"
    )

    return metrics_df, thresholds_df, predictions_df


metrics_df, thresholds_df, predictions_df = fit_acc_similarity(
    job_csv="../data/acc/audit_tax_accounting_jobs.csv",
    job_embeddings_jsonl="../embedding/jobs_train_embeddings.jsonl",
    skill_embeddings_jsonl="../embedding/acc_skills_embeddings.jsonl",
    min_positives_for_skill_tuning=10,
)

print("\nThresholds preview:")
print(thresholds_df.head(10).to_string(index=False))

print("\nPredictions preview:")
print(predictions_df.head(5).to_string(index=False))

Evaluated 59 jobs against 229 ACC skills.
 num_jobs  num_skills  global_threshold  micro_precision  micro_recall  micro_f1  macro_precision  macro_recall  macro_f1  min_positives_for_skill_tuning  num_skill_specific_thresholds
       59         229            0.6489           0.3411        0.7602    0.4709           0.0351        0.0722    0.0456                              10                             15

Global threshold: 0.6489
Skill-specific thresholds used for 15 skills (minimum positive support = 10)

Thresholds preview:
                           skill_name  threshold  positive_support    tuning_type  tp  fp  fn  precision   recall       f1
                   Account Management   0.648934                 0         global   0   7   0   0.000000 0.000000 0.000000
                 Accounting Standards   0.510663                42 skill_specific  42  17   0   0.711864 1.000000 0.831683
           Accounting and Tax Systems   0.648934                 8         global   0   9   8  

## Tune threshold for any skill that improves micro-f1

After obtaining an initial global threshold, we applied greedy coordinate-wise threshold refinement. Each skill threshold was adjusted individually while keeping all others fixed, and the update was retained only if it improved the overall micro-F1 on the evaluation set.

In [11]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics.pairwise import cosine_similarity


def parse_skill_list(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return []

    seen = set()
    skills = []
    for raw in text.split("|"):
        skill = raw.strip()
        if skill and skill not in seen:
            seen.add(skill)
            skills.append(skill)
    return skills


def load_jsonl(path):
    records = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    if not records:
        raise ValueError(f"No records found in {path}")
    return records


def build_indicator_matrix(skill_lists, skill_names):
    skill_to_idx = {skill: idx for idx, skill in enumerate(skill_names)}
    indicator = np.zeros((len(skill_lists), len(skill_names)), dtype=np.uint8)

    for row_idx, skills in enumerate(skill_lists):
        for skill in skills:
            if skill in skill_to_idx:
                indicator[row_idx, skill_to_idx[skill]] = 1

    return indicator


def _safe_div(numerator, denominator):
    numerator = np.asarray(numerator, dtype=np.float64)
    denominator = np.asarray(denominator, dtype=np.float64)
    result = np.zeros_like(numerator, dtype=np.float64)
    np.divide(numerator, denominator, out=result, where=denominator != 0)
    return result


def _binary_f1(tp, fp, fn):
    precision = _safe_div(tp, tp + fp)
    recall = _safe_div(tp, tp + fn)
    return _safe_div(2.0 * precision * recall, precision + recall)


def _micro_metrics_from_counts(tp, fp, fn):
    precision = float(tp / (tp + fp)) if (tp + fp) else 0.0
    recall = float(tp / (tp + fn)) if (tp + fn) else 0.0
    f1 = float((2.0 * precision * recall) / (precision + recall)) if (precision + recall) else 0.0
    return precision, recall, f1


EPS = 1e-12


def _is_better_candidate(candidate, incumbent):
    c_f1, c_recall, c_macro_f1, c_threshold = candidate
    i_f1, i_recall, i_macro_f1, i_threshold = incumbent

    if c_f1 > i_f1 + EPS:
        return True
    if i_f1 > c_f1 + EPS:
        return False

    if c_recall > i_recall + EPS:
        return True
    if i_recall > c_recall + EPS:
        return False

    if c_macro_f1 > i_macro_f1 + EPS:
        return True
    if i_macro_f1 > c_macro_f1 + EPS:
        return False

    return c_threshold < i_threshold - EPS


def _find_best_global_threshold(similarity, labels):
    flat_scores = similarity.ravel()
    flat_labels = labels.ravel().astype(bool)

    candidates = np.unique(flat_scores)
    candidates = np.r_[candidates, flat_scores.max() + 1e-9]

    best = None
    best_threshold = None

    for threshold in candidates:
        pred = flat_scores >= threshold
        tp = int(np.logical_and(pred, flat_labels).sum())
        fp = int(np.logical_and(pred, ~flat_labels).sum())
        fn = int(np.logical_and(~pred, flat_labels).sum())
        _, micro_recall, micro_f1 = _micro_metrics_from_counts(tp, fp, fn)

        pred_2d = similarity >= threshold
        tp_cols = np.logical_and(pred_2d, labels == 1).sum(axis=0)
        fp_cols = np.logical_and(pred_2d, labels == 0).sum(axis=0)
        fn_cols = np.logical_and(~pred_2d, labels == 1).sum(axis=0)
        macro_f1 = float(_binary_f1(tp_cols, fp_cols, fn_cols).mean())

        candidate = (micro_f1, micro_recall, macro_f1, float(threshold))
        if best is None or _is_better_candidate(candidate, best):
            best = candidate
            best_threshold = float(threshold)

    return best_threshold


def _evaluate_micro_f1_with_thresholds(similarity, labels, thresholds):
    predictions = similarity >= thresholds[np.newaxis, :]
    positives = labels == 1

    tp_cols = np.logical_and(predictions, positives).sum(axis=0).astype(np.int64)
    fp_cols = np.logical_and(predictions, ~positives).sum(axis=0).astype(np.int64)
    fn_cols = np.logical_and(~predictions, positives).sum(axis=0).astype(np.int64)

    total_tp = int(tp_cols.sum())
    total_fp = int(fp_cols.sum())
    total_fn = int(fn_cols.sum())

    micro_precision, micro_recall, micro_f1 = _micro_metrics_from_counts(total_tp, total_fp, total_fn)

    return {
        "predictions": predictions,
        "tp_cols": tp_cols,
        "fp_cols": fp_cols,
        "fn_cols": fn_cols,
        "total_tp": total_tp,
        "total_fp": total_fp,
        "total_fn": total_fn,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
    }


def _find_best_threshold_for_one_skill_by_global_micro_f1(
    similarity,
    labels,
    thresholds,
    skill_idx,
):
    current_threshold = float(thresholds[skill_idx])
    candidates = np.unique(similarity[:, skill_idx])
    candidates = np.r_[candidates, similarity[:, skill_idx].max() + 1e-9]

    base_eval = _evaluate_micro_f1_with_thresholds(similarity, labels, thresholds)
    best_micro_f1 = base_eval["micro_f1"]
    best_micro_recall = base_eval["micro_recall"]
    best_threshold = current_threshold

    for threshold in candidates:
        trial_thresholds = thresholds.copy()
        trial_thresholds[skill_idx] = float(threshold)

        trial_eval = _evaluate_micro_f1_with_thresholds(similarity, labels, trial_thresholds)
        trial_micro_f1 = trial_eval["micro_f1"]
        trial_micro_recall = trial_eval["micro_recall"]

        if trial_micro_f1 > best_micro_f1 + EPS:
            best_micro_f1 = trial_micro_f1
            best_micro_recall = trial_micro_recall
            best_threshold = float(threshold)
        elif abs(trial_micro_f1 - best_micro_f1) <= EPS:
            if trial_micro_recall > best_micro_recall + EPS:
                best_micro_recall = trial_micro_recall
                best_threshold = float(threshold)
            elif abs(trial_micro_recall - best_micro_recall) <= EPS:
                if float(threshold) < best_threshold - EPS:
                    best_threshold = float(threshold)

    return best_threshold, best_micro_f1


def _greedy_optimize_thresholds_for_micro_f1(
    similarity,
    labels,
    initial_thresholds,
    max_rounds=3,
):
    thresholds = initial_thresholds.astype(np.float32).copy()
    num_skills = similarity.shape[1]

    initial_eval = _evaluate_micro_f1_with_thresholds(similarity, labels, thresholds)
    best_micro_f1 = initial_eval["micro_f1"]

    threshold_origin = np.array(["global"] * num_skills, dtype=object)

    for round_idx in range(max_rounds):
        improved_this_round = False

        for skill_idx in range(num_skills):
            current_threshold = float(thresholds[skill_idx])

            best_threshold, candidate_micro_f1 = _find_best_threshold_for_one_skill_by_global_micro_f1(
                similarity=similarity,
                labels=labels,
                thresholds=thresholds,
                skill_idx=skill_idx,
            )

            if abs(best_threshold - current_threshold) > EPS and candidate_micro_f1 > best_micro_f1 + EPS:
                thresholds[skill_idx] = best_threshold
                best_micro_f1 = candidate_micro_f1
                threshold_origin[skill_idx] = "greedy_skill_specific"
                improved_this_round = True

        if not improved_this_round:
            break

    return thresholds, threshold_origin


def fit_acc_similarity(
    job_csv="data/acc/audit_tax_accounting_jobs.csv",
    job_embeddings_jsonl="embedding/jobs_train_embeddings.jsonl",
    skill_embeddings_jsonl="embedding/acc_skills_embeddings.jsonl",
    max_greedy_rounds=3,
):
    labels_df = pd.read_csv(job_csv, usecols=["uuid", "title", "extracted_skills"])

    job_records = load_jsonl(job_embeddings_jsonl)
    skill_records = load_jsonl(skill_embeddings_jsonl)

    jobs_df = pd.DataFrame(
        {
            "job_id": [record["job_id"] for record in job_records],
            "embedded_title": [record.get("title", "") for record in job_records],
            "job_embedding": [record["embedding"] for record in job_records],
        }
    )

    merged = jobs_df.merge(
        labels_df,
        left_on="job_id",
        right_on="uuid",
        how="left",
    )

    if merged["uuid"].isna().any():
        missing_count = int(merged["uuid"].isna().sum())
        raise ValueError(f"{missing_count} embedded jobs have no matching labels in the CSV")

    skill_names = [record["skill_name"] for record in skill_records]
    actual_skill_lists = merged["extracted_skills"].map(parse_skill_list).tolist()

    job_matrix = np.vstack(merged["job_embedding"].to_numpy()).astype(np.float32)
    skill_matrix = np.vstack([record["embedding"] for record in skill_records]).astype(np.float32)

    similarity = cosine_similarity(job_matrix, skill_matrix)

    labels = build_indicator_matrix(actual_skill_lists, skill_names)
    if similarity.shape != labels.shape:
        raise ValueError(
            f"Similarity matrix shape {similarity.shape} does not match label matrix shape {labels.shape}"
        )

    global_threshold = _find_best_global_threshold(similarity, labels)
    initial_thresholds = np.full(len(skill_names), global_threshold, dtype=np.float32)

    thresholds, threshold_origin = _greedy_optimize_thresholds_for_micro_f1(
        similarity=similarity,
        labels=labels,
        initial_thresholds=initial_thresholds,
        max_rounds=max_greedy_rounds,
    )

    final_eval = _evaluate_micro_f1_with_thresholds(similarity, labels, thresholds)
    predictions = final_eval["predictions"]
    tp_cols = final_eval["tp_cols"]
    fp_cols = final_eval["fp_cols"]
    fn_cols = final_eval["fn_cols"]

    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    positive_support = labels.sum(axis=0).astype(int)

    metrics_df = pd.DataFrame(
        [
            {
                "num_jobs": len(merged),
                "num_skills": len(skill_names),
                "global_threshold": global_threshold,
                "micro_precision": final_eval["micro_precision"],
                "micro_recall": final_eval["micro_recall"],
                "micro_f1": final_eval["micro_f1"],
                "macro_precision": float(macro_precision),
                "macro_recall": float(macro_recall),
                "macro_f1": float(macro_f1),
                "max_greedy_rounds": max_greedy_rounds,
                "num_skill_specific_thresholds": int((threshold_origin == "greedy_skill_specific").sum()),
            }
        ]
    )

    thresholds_df = pd.DataFrame(
        {
            "skill_name": skill_names,
            "threshold": thresholds.astype(float),
            "positive_support": positive_support,
            "threshold_origin": threshold_origin,
            "tp": tp_cols,
            "fp": fp_cols,
            "fn": fn_cols,
            "precision": _safe_div(tp_cols, tp_cols + fp_cols),
            "recall": _safe_div(tp_cols, tp_cols + fn_cols),
            "f1": _binary_f1(tp_cols, fp_cols, fn_cols),
        }
    )

    predicted_skill_lists = [
        [skill_names[idx] for idx, is_on in enumerate(row) if is_on]
        for row in predictions
    ]

    predictions_df = pd.DataFrame(
        {
            "job_id": merged["job_id"],
            "title": merged["title"].fillna(merged["embedded_title"]).astype(str),
            "actual_skills": [" | ".join(skills) for skills in actual_skill_lists],
            "predicted_skills": [" | ".join(skills) for skills in predicted_skill_lists],
            "tp": [len(set(a) & set(p)) for a, p in zip(actual_skill_lists, predicted_skill_lists)],
            "fp": [len(set(p) - set(a)) for a, p in zip(actual_skill_lists, predicted_skill_lists)],
            "fn": [len(set(a) - set(p)) for a, p in zip(actual_skill_lists, predicted_skill_lists)],
        }
    )

    print(f"Evaluated {len(merged)} jobs against {len(skill_names)} ACC skills.")
    print(metrics_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
    print(f"\nGlobal threshold: {global_threshold:.4f}")
    print(
        f"Greedy skill-specific thresholds kept for "
        f"{int((threshold_origin == 'greedy_skill_specific').sum())} skills"
    )

    return metrics_df, thresholds_df, predictions_df


metrics_df, thresholds_df, predictions_df = fit_acc_similarity(
    job_csv="../data/acc/audit_tax_accounting_jobs.csv",
    job_embeddings_jsonl="../embedding/jobs_train_embeddings.jsonl",
    skill_embeddings_jsonl="../embedding/acc_skills_embeddings.jsonl",
    max_greedy_rounds=3,
)

print("\nThresholds preview:")
print(thresholds_df.head(10).to_string(index=False))

print("\nPredictions preview:")
print(predictions_df.head(5).to_string(index=False))

Evaluated 59 jobs against 229 ACC skills.
 num_jobs  num_skills  global_threshold  micro_precision  micro_recall  micro_f1  macro_precision  macro_recall  macro_f1  max_greedy_rounds  num_skill_specific_thresholds
       59         229            0.6489           0.4646        0.7285    0.5674           0.0563        0.0733    0.0555                  3                             75

Global threshold: 0.6489
Greedy skill-specific thresholds kept for 75 skills

Thresholds preview:
                           skill_name  threshold  positive_support      threshold_origin  tp  fp  fn  precision  recall       f1
                   Account Management   0.701580                 0 greedy_skill_specific   0   1   0   0.000000     0.0 0.000000
                 Accounting Standards   0.510663                42 greedy_skill_specific  42  17   0   0.711864     1.0 0.831683
           Accounting and Tax Systems   0.719695                 8 greedy_skill_specific   0   1   8   0.000000     0.0 0.000000

## Comparing results of all 3 models

Evaluated 59 jobs against 229 ACC skills.

Only Global Threshold

| global_threshold | micro_precision | micro_recall | micro_f1 | macro_precision | macro_recall | macro_f1 |
|---:|---:|---:|---:|---:|---:|---:|
| 0.6489 | 0.1375 | 0.1244 | 0.1306 | 0.0412 | 0.0196 | 0.0179 |

Calculated Threshold for 15 Common Skills

| global_threshold | micro_precision | micro_recall | micro_f1 | macro_precision | macro_recall | macro_f1 | min_positives_for_skill_tuning | num_skill_specific_thresholds |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 0.6489 | 0.3411 | 0.7602 | 0.4709 | 0.0351 | 0.0722 | 0.0456 | 10 | 15 |

3rd Model

| global_threshold | micro_precision | micro_recall | micro_f1 | macro_precision | macro_recall | macro_f1 | max_greedy_rounds | num_skill_specific_thresholds |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 0.6489 | 0.4646 | 0.7285 | 0.5674 | 0.0563 | 0.0733 | 0.0555 | 3 | 75 |

# Use both jobs and courses

This section reuses the same thresholding logic as the jobs-only models, but expands the labeled entity set by appending ACC courses to the training examples before computing cosine similarity against the ACC skill embeddings.


At a high level, it does this:

1. Load entities like jobs or courses together with their true labeled skills
2. Load embeddings for those entities and for ACC skills
3. Compute cosine similarity between every entity and every skill
4. Turn the true skill lists into a binary label matrix
5. Try different thresholding strategies to convert similarity scores into predicted skills
6. Output evaluation tables and per-entity predictions


In [17]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics.pairwise import cosine_similarity

required_helpers = [
    "load_jsonl",
    "parse_skill_list",
    "build_indicator_matrix",
    "_safe_div",
    "_binary_f1",
    "_micro_metrics_from_counts",
    "_find_best_global_threshold",
    "_find_best_threshold_for_one_skill",
    "_greedy_optimize_thresholds_for_micro_f1",
]
missing_helpers = [name for name in required_helpers if name not in globals()]
if missing_helpers:
    raise NameError(
        "Run the earlier jobs-only helper cells before this section. Missing helpers: "
        + ", ".join(missing_helpers)
    )


def load_labeled_entities_from_embeddings(label_csv, embeddings_jsonl, *, entity_type, entity_id_col, embedding_id_key):
    labels_df = pd.read_csv(label_csv, usecols=[entity_id_col, "title", "extracted_skills"]).rename(
        columns={entity_id_col: "entity_id", "title": "label_title"}
    )
    embedding_records = load_jsonl(embeddings_jsonl)

    embeddings_df = pd.DataFrame(
        {
            "entity_type": entity_type,
            "entity_id": [record[embedding_id_key] for record in embedding_records],
            "embedded_title": [record.get("title", "") for record in embedding_records],
            "embedding": [record["embedding"] for record in embedding_records],
        }
    )

    merged = embeddings_df.merge(labels_df, on="entity_id", how="left")
    missing_labels = int(merged["label_title"].isna().sum())
    if missing_labels:
        raise ValueError(f"{missing_labels} embedded {entity_type} rows have no matching labels in {label_csv}")

    merged["title"] = merged["label_title"].fillna(merged["embedded_title"]).astype(str)
    merged["actual_skill_lists"] = merged["extracted_skills"].map(parse_skill_list)

    summary = {
        "entity_type": entity_type,
        "embedded_rows": len(embedding_records),
        "matched_labels": len(merged),
        "missing_labels": missing_labels,
        "rows_with_extracted_skills": int(merged["actual_skill_lists"].map(bool).sum()),
    }

    return merged[["entity_type", "entity_id", "title", "embedding", "actual_skill_lists"]].copy(), summary


def prepare_acc_similarity_problem(entities_df, skill_embeddings_jsonl):
    skill_records = load_jsonl(skill_embeddings_jsonl)
    skill_names = [record["skill_name"] for record in skill_records]

    entity_matrix = np.vstack(entities_df["embedding"].to_numpy()).astype(np.float32)
    skill_matrix = np.vstack([record["embedding"] for record in skill_records]).astype(np.float32)

    similarity = cosine_similarity(entity_matrix, skill_matrix)
    labels = build_indicator_matrix(entities_df["actual_skill_lists"].tolist(), skill_names)

    if similarity.shape != labels.shape:
        raise ValueError(
            f"Similarity matrix shape {similarity.shape} does not match label matrix shape {labels.shape}"
        )

    return {
        "entities_df": entities_df.reset_index(drop=True).copy(),
        "skill_names": skill_names,
        "similarity": similarity,
        "labels": labels,
    }


def _build_result_frames(problem, thresholds, predictions, global_threshold, *, dataset_variant, model_variant, metric_extras=None, threshold_extras=None):
    entities_df = problem["entities_df"]
    skill_names = problem["skill_names"]
    labels = problem["labels"]

    positives = labels == 1
    tp_cols = np.logical_and(predictions, positives).sum(axis=0).astype(np.int64)
    fp_cols = np.logical_and(predictions, ~positives).sum(axis=0).astype(np.int64)
    fn_cols = np.logical_and(~predictions, positives).sum(axis=0).astype(np.int64)

    total_tp = int(tp_cols.sum())
    total_fp = int(fp_cols.sum())
    total_fn = int(fn_cols.sum())

    micro_precision, micro_recall, micro_f1 = _micro_metrics_from_counts(total_tp, total_fp, total_fn)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    counts = entities_df["entity_type"].value_counts()
    positive_support = labels.sum(axis=0).astype(int)

    metrics_record = {
        "dataset_variant": dataset_variant,
        "model_variant": model_variant,
        "num_jobs": int(counts.get("job", 0)),
        "num_courses": int(counts.get("course", 0)),
        "num_entities": len(entities_df),
        "num_skills": len(skill_names),
        "global_threshold": float(global_threshold),
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
    }
    if metric_extras:
        metrics_record.update(metric_extras)

    thresholds_data = {
        "dataset_variant": dataset_variant,
        "model_variant": model_variant,
        "skill_name": skill_names,
        "threshold": thresholds.astype(float),
        "positive_support": positive_support,
        "tp": tp_cols,
        "fp": fp_cols,
        "fn": fn_cols,
        "precision": _safe_div(tp_cols, tp_cols + fp_cols),
        "recall": _safe_div(tp_cols, tp_cols + fn_cols),
        "f1": _binary_f1(tp_cols, fp_cols, fn_cols),
    }
    if threshold_extras:
        thresholds_data.update(threshold_extras)

    predicted_skill_lists = [
        [skill_names[idx] for idx, is_on in enumerate(row) if is_on]
        for row in predictions
    ]
    actual_skill_lists = entities_df["actual_skill_lists"].tolist()

    predictions_df = pd.DataFrame(
        {
            "entity_type": entities_df["entity_type"],
            "entity_id": entities_df["entity_id"],
            "title": entities_df["title"].astype(str),
            "actual_skills": [" | ".join(skills) for skills in actual_skill_lists],
            "predicted_skills": [" | ".join(skills) for skills in predicted_skill_lists],
            "tp": [len(set(a) & set(p)) for a, p in zip(actual_skill_lists, predicted_skill_lists)],
            "fp": [len(set(p) - set(a)) for a, p in zip(actual_skill_lists, predicted_skill_lists)],
            "fn": [len(set(a) - set(p)) for a, p in zip(actual_skill_lists, predicted_skill_lists)],
        }
    )

    return pd.DataFrame([metrics_record]), pd.DataFrame(thresholds_data), predictions_df


def fit_global_threshold_model(problem, *, dataset_variant):
    similarity = problem["similarity"]
    labels = problem["labels"]
    global_threshold = _find_best_global_threshold(similarity, labels)
    thresholds = np.full(labels.shape[1], global_threshold, dtype=np.float32)
    predictions = similarity >= global_threshold
    return _build_result_frames(
        problem,
        thresholds,
        predictions,
        global_threshold,
        dataset_variant=dataset_variant,
        model_variant="global_threshold",
    )


def fit_common_skill_threshold_model(problem, *, dataset_variant, min_positives_for_skill_tuning=10):
    similarity = problem["similarity"]
    labels = problem["labels"]
    global_threshold = _find_best_global_threshold(similarity, labels)

    positive_support = labels.sum(axis=0).astype(int)
    thresholds = np.full(labels.shape[1], global_threshold, dtype=np.float32)
    tuning_type = np.array(["global"] * labels.shape[1], dtype=object)

    for skill_idx in range(labels.shape[1]):
        if positive_support[skill_idx] >= min_positives_for_skill_tuning:
            thresholds[skill_idx] = _find_best_threshold_for_one_skill(
                scores=similarity[:, skill_idx],
                labels=labels[:, skill_idx],
                default_threshold=global_threshold,
            )
            tuning_type[skill_idx] = "skill_specific"

    predictions = similarity >= thresholds[np.newaxis, :]
    return _build_result_frames(
        problem,
        thresholds,
        predictions,
        global_threshold,
        dataset_variant=dataset_variant,
        model_variant="common_skill_tuning",
        metric_extras={
            "min_positives_for_skill_tuning": min_positives_for_skill_tuning,
            "num_skill_specific_thresholds": int((tuning_type == "skill_specific").sum()),
        },
        threshold_extras={"tuning_type": tuning_type},
    )


def fit_greedy_threshold_model(problem, *, dataset_variant, max_greedy_rounds=3):
    similarity = problem["similarity"]
    labels = problem["labels"]
    global_threshold = _find_best_global_threshold(similarity, labels)
    initial_thresholds = np.full(labels.shape[1], global_threshold, dtype=np.float32)

    thresholds, threshold_origin = _greedy_optimize_thresholds_for_micro_f1(
        similarity=similarity,
        labels=labels,
        initial_thresholds=initial_thresholds,
        max_rounds=max_greedy_rounds,
    )

    predictions = similarity >= thresholds[np.newaxis, :]
    return _build_result_frames(
        problem,
        thresholds,
        predictions,
        global_threshold,
        dataset_variant=dataset_variant,
        model_variant="greedy_micro_f1",
        metric_extras={
            "max_greedy_rounds": max_greedy_rounds,
            "num_skill_specific_thresholds": int((threshold_origin == "greedy_skill_specific").sum()),
        },
        threshold_extras={"threshold_origin": threshold_origin},
    )


def run_all_models_for_dataset(problem, *, dataset_variant, min_positives_for_skill_tuning=10, max_greedy_rounds=3):
    return {
        "global_threshold": fit_global_threshold_model(problem, dataset_variant=dataset_variant),
        "common_skill_tuning": fit_common_skill_threshold_model(
            problem,
            dataset_variant=dataset_variant,
            min_positives_for_skill_tuning=min_positives_for_skill_tuning,
        ),
        "greedy_micro_f1": fit_greedy_threshold_model(
            problem,
            dataset_variant=dataset_variant,
            max_greedy_rounds=max_greedy_rounds,
        ),
    }


def evaluate_fixed_threshold_model(problem, *, thresholds_df, dataset_variant, model_variant, global_threshold, metric_extras=None, threshold_extras=None):
    required_columns = {"skill_name", "threshold"}
    missing_columns = required_columns.difference(thresholds_df.columns)
    if missing_columns:
        raise ValueError(
            "thresholds_df is missing required columns: "
            + ", ".join(sorted(missing_columns))
        )

    duplicate_skill_names = thresholds_df.loc[thresholds_df["skill_name"].duplicated(), "skill_name"].tolist()
    if duplicate_skill_names:
        raise ValueError(
            "thresholds_df has duplicate skill_name values: "
            + ", ".join(map(str, duplicate_skill_names[:10]))
            + (" ..." if len(duplicate_skill_names) > 10 else "")
        )

    skill_names = problem["skill_names"]
    aligned_thresholds_df = thresholds_df.set_index("skill_name").reindex(skill_names)

    missing_skill_names = aligned_thresholds_df.index[aligned_thresholds_df["threshold"].isna()].tolist()
    if missing_skill_names:
        raise ValueError(
            "Learned threshold table is missing thresholds for validation skills: "
            + ", ".join(map(str, missing_skill_names[:10]))
            + (" ..." if len(missing_skill_names) > 10 else "")
        )

    thresholds = aligned_thresholds_df["threshold"].to_numpy(dtype=np.float32)
    predictions = problem["similarity"] >= thresholds[np.newaxis, :]

    resolved_threshold_extras = dict(threshold_extras or {})
    if "threshold_origin" in aligned_thresholds_df.columns and "threshold_origin" not in resolved_threshold_extras:
        resolved_threshold_extras["threshold_origin"] = aligned_thresholds_df["threshold_origin"].fillna("global").to_numpy(dtype=object)

    return _build_result_frames(
        problem,
        thresholds,
        predictions,
        global_threshold,
        dataset_variant=dataset_variant,
        model_variant=model_variant,
        metric_extras=metric_extras,
        threshold_extras=resolved_threshold_extras or None,
    )


In [18]:
job_entities, job_join_summary = load_labeled_entities_from_embeddings(
    "../data/acc/audit_tax_accounting_jobs.csv",
    "../embedding/jobs_train_embeddings.jsonl",
    entity_type="job",
    entity_id_col="uuid",
    embedding_id_key="job_id",
)
course_entities, course_join_summary = load_labeled_entities_from_embeddings(
    "../data/acc/acc_courses.csv",
    "../embedding/courses_train_embeddings.jsonl",
    entity_type="course",
    entity_id_col="moduleCode",
    embedding_id_key="course_code",
)

join_validation_df = pd.DataFrame([job_join_summary, course_join_summary])
jobs_only_problem = prepare_acc_similarity_problem(job_entities, "../embedding/acc_skills_embeddings.jsonl")
jobs_plus_courses_problem = prepare_acc_similarity_problem(
    pd.concat([job_entities, course_entities], ignore_index=True),
    "../embedding/acc_skills_embeddings.jsonl",
)
shape_validation_df = pd.DataFrame(
    [
        {
            "dataset_variant": "jobs_only",
            "similarity_shape": jobs_only_problem["similarity"].shape,
            "label_shape": jobs_only_problem["labels"].shape,
        },
        {
            "dataset_variant": "jobs_plus_courses",
            "similarity_shape": jobs_plus_courses_problem["similarity"].shape,
            "label_shape": jobs_plus_courses_problem["labels"].shape,
        },
    ]
)

jobs_only_runs = run_all_models_for_dataset(jobs_only_problem, dataset_variant="jobs_only")
jobs_plus_courses_runs = run_all_models_for_dataset(
    jobs_plus_courses_problem,
    dataset_variant="jobs_plus_courses",
)

comparison_df = pd.concat(
    [
        jobs_only_runs["global_threshold"][0],
        jobs_only_runs["common_skill_tuning"][0],
        jobs_only_runs["greedy_micro_f1"][0],
        jobs_plus_courses_runs["global_threshold"][0],
        jobs_plus_courses_runs["common_skill_tuning"][0],
        jobs_plus_courses_runs["greedy_micro_f1"][0],
    ],
    ignore_index=True,
)[
    [
        "dataset_variant",
        "model_variant",
        "num_jobs",
        "num_courses",
        "num_entities",
        "num_skills",
        "global_threshold",
        "micro_precision",
        "micro_recall",
        "micro_f1",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "min_positives_for_skill_tuning",
        "max_greedy_rounds",
        "num_skill_specific_thresholds",
    ]
]

combined_global_metrics_df, combined_global_thresholds_df, combined_global_predictions_df = jobs_plus_courses_runs["global_threshold"]
combined_common_metrics_df, combined_common_thresholds_df, combined_common_predictions_df = jobs_plus_courses_runs["common_skill_tuning"]
combined_greedy_metrics_df, combined_greedy_thresholds_df, combined_greedy_predictions_df = jobs_plus_courses_runs["greedy_micro_f1"]

combined_predictions_preview_df = pd.concat(
    [
        combined_greedy_predictions_df.query("entity_type == 'job'").head(3),
        combined_greedy_predictions_df.query("entity_type == 'course'").head(3),
    ],
    ignore_index=True,
)

print("Join validation:")
print(join_validation_df.to_string(index=False))

print()
print("Problem shapes:")
print(shape_validation_df.to_string(index=False))

print()
print("Comparison across jobs-only vs jobs+courses:")
print(comparison_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

print()
print("Combined greedy thresholds preview:")
print(combined_greedy_thresholds_df.head(10).to_string(index=False))

print()
print("Combined greedy predictions preview:")
print(
    combined_predictions_preview_df[
        ["entity_type", "entity_id", "title", "actual_skills", "predicted_skills", "tp", "fp", "fn"]
    ].to_string(index=False)
)


Join validation:
entity_type  embedded_rows  matched_labels  missing_labels  rows_with_extracted_skills
        job             59              59               0                          59
     course             38              38               0                          35

Problem shapes:
  dataset_variant similarity_shape label_shape
        jobs_only        (59, 229)   (59, 229)
jobs_plus_courses        (97, 229)   (97, 229)

Comparison across jobs-only vs jobs+courses:
  dataset_variant       model_variant  num_jobs  num_courses  num_entities  num_skills  global_threshold  micro_precision  micro_recall  micro_f1  macro_precision  macro_recall  macro_f1  min_positives_for_skill_tuning  max_greedy_rounds  num_skill_specific_thresholds
        jobs_only    global_threshold        59            0            59         229            0.6489           0.1375        0.1244    0.1306           0.0412        0.0196    0.0179                             NaN                NaN            

# Run the best threshold on the validation set

In [19]:
required_validation_globals = [
    "combined_greedy_metrics_df",
    "combined_greedy_thresholds_df",
]
missing_validation_globals = [name for name in required_validation_globals if name not in globals()]
if missing_validation_globals:
    raise NameError(
        "Run the previous jobs_plus_courses training cell before this validation cell. Missing globals: "
        + ", ".join(missing_validation_globals)
    )

val_job_entities, val_job_join_summary = load_labeled_entities_from_embeddings(
    "../data/acc/audit_tax_accounting_jobs.csv",
    "../embedding/jobs_val_embeddings.jsonl",
    entity_type="job",
    entity_id_col="uuid",
    embedding_id_key="job_id",
)
val_course_entities, val_course_join_summary = load_labeled_entities_from_embeddings(
    "../data/acc/acc_courses.csv",
    "../embedding/courses_val_embeddings.jsonl",
    entity_type="course",
    entity_id_col="moduleCode",
    embedding_id_key="course_code",
)

val_join_validation_df = pd.DataFrame([val_job_join_summary, val_course_join_summary])
jobs_plus_courses_val_problem = prepare_acc_similarity_problem(
    pd.concat([val_job_entities, val_course_entities], ignore_index=True),
    "../embedding/acc_skills_embeddings.jsonl",
)
val_shape_validation_df = pd.DataFrame(
    [
        {
            "dataset_variant": "jobs_plus_courses_val",
            "similarity_shape": jobs_plus_courses_val_problem["similarity"].shape,
            "label_shape": jobs_plus_courses_val_problem["labels"].shape,
        }
    ]
)

val_threshold_source_global_threshold = float(combined_greedy_metrics_df.loc[0, "global_threshold"])
val_greedy_metrics_df, val_greedy_thresholds_df, val_greedy_predictions_df = evaluate_fixed_threshold_model(
    jobs_plus_courses_val_problem,
    thresholds_df=combined_greedy_thresholds_df,
    dataset_variant="jobs_plus_courses_val",
    model_variant="greedy_micro_f1",
    global_threshold=val_threshold_source_global_threshold,
    metric_extras={
        "threshold_source_dataset_variant": "jobs_plus_courses",
        "threshold_source_model_variant": "greedy_micro_f1",
    },
    threshold_extras={
        "threshold_source_dataset_variant": "jobs_plus_courses",
        "threshold_source_model_variant": "greedy_micro_f1",
    },
)

aligned_train_thresholds_df = (
    combined_greedy_thresholds_df
    .set_index("skill_name")
    .loc[val_greedy_thresholds_df["skill_name"], ["threshold", "threshold_origin"]]
    .reset_index()
)

if not np.allclose(
    val_greedy_thresholds_df["threshold"].to_numpy(dtype=np.float32),
    aligned_train_thresholds_df["threshold"].to_numpy(dtype=np.float32),
):
    raise AssertionError("Validation thresholds do not match the aligned train greedy thresholds")

if not val_greedy_thresholds_df["threshold_origin"].equals(aligned_train_thresholds_df["threshold_origin"]):
    raise AssertionError("Validation threshold_origin does not match the aligned train greedy thresholds")

val_predictions_preview_df = pd.concat(
    [
        val_greedy_predictions_df.query("entity_type == 'job'").head(3),
        val_greedy_predictions_df.query("entity_type == 'course'").head(3),
    ],
    ignore_index=True,
)

print("Validation join summary:")
print(val_join_validation_df.to_string(index=False))

print()
print("Validation problem shape:")
print(val_shape_validation_df.to_string(index=False))

print()
print("Validation greedy metrics:")
print(val_greedy_metrics_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

print()
print("Validation greedy thresholds preview:")
print(val_greedy_thresholds_df.head(10).to_string(index=False))

print()
print("Validation greedy predictions preview:")
print(
    val_predictions_preview_df[
        ["entity_type", "entity_id", "title", "actual_skills", "predicted_skills", "tp", "fp", "fn"]
    ].to_string(index=False)
)

print()
print("Threshold alignment check: passed")
print("Threshold origin preservation check: passed")


Validation join summary:
entity_type  embedded_rows  matched_labels  missing_labels  rows_with_extracted_skills
        job             19              19               0                          19
     course             13              13               0                          13

Validation problem shape:
      dataset_variant similarity_shape label_shape
jobs_plus_courses_val        (32, 229)   (32, 229)

Validation greedy metrics:
      dataset_variant   model_variant  num_jobs  num_courses  num_entities  num_skills  global_threshold  micro_precision  micro_recall  micro_f1  macro_precision  macro_recall  macro_f1 threshold_source_dataset_variant threshold_source_model_variant
jobs_plus_courses_val greedy_micro_f1        19           13            32         229            0.5382           0.1816        0.6736    0.2861           0.0230        0.0565    0.0299                jobs_plus_courses                greedy_micro_f1

Validation greedy thresholds preview:
      dataset_va